# BrainScan-v2 — Étape 1 : Classification des tumeurs cérébrales (GoogLeNet)

## 1.4 — Entraînement, évaluation, matrice de confusion et suivi MLflow

**Objectifs du notebook**

1. Entraîner le modèle GoogLeNet sur l'ensemble d'entraînement en suivant les
   performances sur l'ensemble de validation.
2. Réaliser un **fine-tuning** (phase B) après la phase d'extraction de caractéristiques.
3. Évaluer avec les métriques demandées : **accuracy, precision, recall, F1-score**.
4. Analyser les résultats avec une **matrice de confusion**.
5. **Comparer les performances sur les ensembles d'entraînement et de test**
   (diagnostic de surapprentissage).
6. **Sauvegarder le modèle entraîné** et les artefacts associés.
7. Enregistrer l'expérience dans **MLflow** (hyperparamètres, métriques, artefacts, modèle).

In [ ]:
# Cellule 0 — Imports, configuration et chemins du projet
"""Cellules communes réutilisées par les 4 notebooks de l'étape 1 (classification)."""

import json
import os
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

import matplotlib

if matplotlib.get_backend().lower() == "agg":  # exécution headless (Docker / nbconvert)
    try:
        matplotlib.use("module://matplotlib_inline.backend_inline")
    except Exception:
        pass

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from tensorflow import keras

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_rows", 200)
pd.set_option("display.max_columns", 50)

# --------------------------------------------------------------------------------------
# Configuration du projet
# --------------------------------------------------------------------------------------
def _detect_project_root() -> Path:
    """Racine du projet (compatible exécution locale et exécution dans Docker /app)."""
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "data" / "Data").is_dir() and (candidate / "src").is_dir():
            return candidate
    if Path("/app/data/Data").is_dir():
        return Path("/app")
    raise FileNotFoundError("Dataset introuvable : « data/Data » manquant depuis " + str(Path.cwd()))


PROJECT_ROOT = _detect_project_root()
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "Data"
PROCESSED_DIR = DATA_DIR / "processed" / "classification"
ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = ARTIFACTS_DIR / "reports"

for _d in (ARTIFACTS_DIR, MODELS_DIR, REPORTS_DIR):
    _d.mkdir(parents=True, exist_ok=True)

if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

#: Graine aléatoire globale (reproductibilité de bout en bout).
SEED = 42

#: Extensions d'images autorisées par le cahier des charges.
ALLOWED_EXTENSIONS = {".jpeg", ".jpg", ".bmp", ".png"}

#: Libellés lisibles pour l'affichage (correspondance dossier -> pathologie).
CLASS_DISPLAY = {
    "glioma": "Gliome",
    "meningioma": "Méningiome",
    "notumor": "Aucune tumeur",
    "pituitary": "Hypophyse",
}

#: Classes du dataset, déduites des noms de dossiers (ordre alphabétique).
class_names = sorted(p.name for p in RAW_DIR.iterdir() if p.is_dir())

print(f"PROJECT_ROOT    : {PROJECT_ROOT}")
print(f"RAW_DIR         : {RAW_DIR}")
print(f"PROCESSED_DIR   : {PROCESSED_DIR}")
print(f"ARTIFACTS_DIR   : {ARTIFACTS_DIR}")
print(f"MODELS_DIR      : {MODELS_DIR}")
print(f"Classes         : {class_names}")


### 1.26 — Rechargement des données et du modèle

Le notebook est autonome : il reconstruit le modèle (avec les poids ImageNet) puis
recharge le dataset nettoyé produit par le notebook 2.

In [ ]:
# Cellule 4.1 — Imports et configuration
import json
import os
import time

import mlflow
import mlflow.keras
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix

from brainscan.training.googlenet import build_googlenet, set_reproducible

set_reproducible(SEED)

MLFLOW_TRACKING_URI = os.environ.get("MLFLOW_TRACKING_URI", "http://localhost:5000")
MLFLOW_EXPERIMENT_NAME = os.environ.get("MLFLOW_EXPERIMENT_NAME", "brainscan-v2")

IMAGE_SIZE = 224
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

print(f"MLFLOW_TRACKING_URI : {MLFLOW_TRACKING_URI}")
print(f"Experiment          : {MLFLOW_EXPERIMENT_NAME}")


In [ ]:
# Cellule 4.2 — Chargement des trois ensembles
def load_dataset(split: str, shuffle: bool, seed: int | None = None) -> tf.data.Dataset:
    ds = tf.keras.utils.image_dataset_from_directory(
        str(PROCESSED_DIR / split),
        labels="inferred",
        label_mode="categorical",
        image_size=(IMAGE_SIZE, IMAGE_SIZE),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        seed=seed,
        color_mode="rgb",
        interpolation="bilinear",
    )
    return ds.cache().prefetch(AUTOTUNE)


train_ds = load_dataset("train", shuffle=True, seed=SEED)
valid_ds = load_dataset("valid", shuffle=False)
test_ds = load_dataset("test", shuffle=False)

for name, ds in [("train", train_ds), ("valid", valid_ds), ("test", test_ds)]:
    print(f"{name:<6s} : {int(tf.data.experimental.cardinality(ds)):>4d} lots")


In [ ]:
# Cellule 4.3 — Reconstruction du modèle
model = build_googlenet(
    num_classes=len(class_names),
    class_names=class_names,
    input_shape=(IMAGE_SIZE, IMAGE_SIZE, 3),
    image_size=IMAGE_SIZE,
    weights="imagenet",
    freeze_backbone=True,
    dense_units=128,
    dropout=0.5,
    include_preprocessing=True,
    augment=True,
)
model.class_names = class_names
print("Modèle reconstruit avec les poids ImageNet.")


### 1.27 — Entraînement — Phase A (backbone gelé)

Callbacks utilisés :

* `EarlyStopping(monitor="val_loss", patience=3, restore_best_weights=True)` — arrêt anticipé ;
* `ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2)` — décroissance du LR ;
* `ModelCheckpoint` — sauvegarde du meilleur modèle sur l'ensemble de validation.

In [ ]:
# Cellule 4.4 — Callbacks
CHECKPOINT_DIR = ARTIFACTS_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

CALLBACKS = [
    keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=3,
        min_delta=1e-4,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
        verbose=1,
    ),
    keras.callbacks.ModelCheckpoint(
        filepath=str(CHECKPOINT_DIR / "googlenet_phaseA.keras"),
        monitor="val_loss",
        save_best_only=True,
        verbose=0,
    ),
]
print("Callbacks configurés : EarlyStopping, ReduceLROnPlateau, ModelCheckpoint")


In [ ]:
# Cellule 4.5 — Phase A : entraînement avec le backbone gelé
LR_PHASE_A = 1e-3
EPOCHS_A = 15

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_PHASE_A),
    loss="categorical_crossentropy",
    metrics=[
        keras.metrics.CategoricalAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.F1Score(name="f1", average="macro"),
    ],
)

t0 = time.time()
history_a = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS_A,
    callbacks=CALLBACKS,
    verbose=2,
)
phase_a_seconds = time.time() - t0

print(f"\nPhase A terminée en {phase_a_seconds / 60:.1f} min")
print(f"val_loss   : {history_a.history['val_loss'][-1]:.4f}")
print(f"val_accuracy: {history_a.history['val_accuracy'][-1]:.4f}")


### 1.28 — Phase B : fine-tuning (backbone dégelé)

Les couches convolutionnelles sont dégelées et le learning rate est divisé par 100
(`1e-5`) afin de préserver les features ImageNet tout en les adaptant au domaine
médical.

In [ ]:
# Cellule 4.6 — Phase B : dégel du backbone et fine-tuning
LR_PHASE_B = 1e-5
EPOCHS_B = 8

for layer in model.layers:
    if layer.name.startswith(("conv", "inception", "maxpool")):
        layer.trainable = True

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LR_PHASE_B),
    loss="categorical_crossentropy",
    metrics=[
        keras.metrics.CategoricalAccuracy(name="accuracy"),
        keras.metrics.Precision(name="precision"),
        keras.metrics.Recall(name="recall"),
        keras.metrics.F1Score(name="f1", average="macro"),
    ],
)

callbacks_b = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=3, min_delta=1e-4,
                                   restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2,
                                       min_lr=1e-7, verbose=1),
    keras.callbacks.ModelCheckpoint(filepath=str(CHECKPOINT_DIR / "googlenet_phaseB.keras"),
                                    monitor="val_loss", save_best_only=True, verbose=0),
]

t0 = time.time()
history_b = model.fit(
    train_ds,
    validation_data=valid_ds,
    epochs=EPOCHS_B,
    callbacks=callbacks_b,
    verbose=2,
)
phase_b_seconds = time.time() - t0

# Historique consolidé
history = {k: history_a.history[k] + history_b.history[k] for k in history_a.history}
print(f"\nPhase B terminée en {phase_b_seconds / 60:.1f} min")


### 1.29 — Courbes d'apprentissage

In [ ]:
# Cellule 4.7 — Courbes d'apprentissage
epochs = np.arange(1, len(history["loss"]) + 1)
phase_a_len = len(history_a.history["loss"])

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(epochs, history["loss"], label="train", color="tab:blue", marker="o", ms=3)
axes[0].plot(epochs, history["val_loss"], label="validation", color="tab:orange", marker="o", ms=3)
axes[0].axvline(phase_a_len, color="grey", ls="--", lw=1)
axes[0].set_title("Perte (loss)", weight="bold")
axes[0].set_xlabel("Époque")
axes[0].legend()

axes[1].plot(epochs, history["accuracy"], label="train", color="tab:blue", marker="o", ms=3)
axes[1].plot(epochs, history["val_accuracy"], label="validation", color="tab:orange", marker="o", ms=3)
axes[1].axvline(phase_a_len, color="grey", ls="--", lw=1)
axes[1].set_title("Accuracy", weight="bold")
axes[1].set_xlabel("Époque")
axes[1].legend()

axes[2].plot(epochs, history["f1"], label="train (F1 macro)", color="tab:blue", marker="o", ms=3)
axes[2].plot(epochs, history["val_f1"], label="validation (F1 macro)", color="tab:orange", marker="o", ms=3)
axes[2].axvline(phase_a_len, color="grey", ls="--", lw=1)
axes[2].set_title("F1-score macro", weight="bold")
axes[2].set_xlabel("Époque")
axes[2].legend()

for ax in axes:
    ax.grid(alpha=0.3)

fig.suptitle("Courbes d'apprentissage — GoogLeNet (trait pointillé = fin de la phase A)",
             fontsize=13, weight="bold")
plt.tight_layout()
plt.show()


### 1.30 — Évaluation : accuracy, precision, recall, F1-score

Les quatre métriques sont calculées sur les trois ensembles (train, validation, test) :
`keras.metrics` en phase d'évaluation, puis recalculées à partir des prédictions avec
`sklearn` (contrôle croisé).

In [ ]:
# Cellule 4.8 — Évaluation sur train / validation / test
def evaluate_split(model, ds, name: str) -> dict[str, float]:
    results = model.evaluate(ds, verbose=0, return_dict=True)
    return {
        "accuracy": float(results["accuracy"]),
        "precision": float(results["precision"]),
        "recall": float(results["recall"]),
        "f1": float(results["f1"]),
        "loss": float(results["loss"]),
    }


eval_metrics = {
    "train": evaluate_split(model, train_ds, "train"),
    "valid": evaluate_split(model, valid_ds, "valid"),
    "test": evaluate_split(model, test_ds, "test"),
}

eval_df = pd.DataFrame(eval_metrics).T[["loss", "accuracy", "precision", "recall", "f1"]]
eval_df = eval_df.rename(index={"valid": "validation"})
eval_df.round(4)


In [ ]:
# Cellule 4.9 — Vérification croisée avec scikit-learn (prédictions + rapport)
def predict_split(model, ds) -> tuple[np.ndarray, np.ndarray]:
    y_true, y_pred = [], []
    for xb, yb in ds:
        probs = model.predict(xb, verbose=0)
        y_true.append(np.argmax(yb.numpy(), axis=1))
        y_pred.append(np.argmax(probs, axis=1))
    return np.concatenate(y_true), np.concatenate(y_pred)


y_true, y_pred = predict_split(model, test_ds)
print("Rapport de classification — ensemble de TEST")
print(classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0))


### 1.31 — Matrice de confusion

In [ ]:
# Cellule 4.10 — Matrice de confusion (test)
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))))
cm_norm = confusion_matrix(y_true, y_pred, labels=list(range(len(class_names))), normalize="true")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names,
            ax=axes[0], cbar=False)
axes[0].set_title("Matrice de confusion — TEST (effectifs)", weight="bold")
axes[0].set_xlabel("Classe prédite")
axes[0].set_ylabel("Classe réelle")

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues", xticklabels=class_names, yticklabels=class_names,
            ax=axes[1], vmin=0, vmax=1)
axes[1].set_title("Matrice de confusion — TEST (normalisée par ligne)", weight="bold")
axes[1].set_xlabel("Classe prédite")
axes[1].set_ylabel("Classe réelle")

plt.tight_layout()
plt.show()


In [ ]:
# Cellule 4.11 — Matrice de confusion (entraînement) et comparaison
y_true_train, y_pred_train = predict_split(model, train_ds)
cm_train = confusion_matrix(y_true_train, y_pred_train, labels=list(range(len(class_names))))

fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(cm_train, annot=True, fmt="d", cmap="Greens", xticklabels=class_names, yticklabels=class_names,
            ax=ax, cbar=False)
ax.set_title("Matrice de confusion — TRAIN (effectifs)", weight="bold")
ax.set_xlabel("Classe prédite")
ax.set_ylabel("Classe réelle")
plt.tight_layout()
plt.show()


### 1.32 — Comparaison entraînement / test (diagnostic de surapprentissage)

L'écart entre les performances sur l'ensemble d'entraînement et sur l'ensemble de test
mesure le **surapprentissage** :

* écart faible (< 3 points) → généralisation correcte ;
* écart important → surapprentissage (renforcer l'augmentation / le dropout).

In [ ]:
# Cellule 4.12 — Analyse de l'écart train / test
gap_rows = []
for metric in ["accuracy", "precision", "recall", "f1"]:
    tr, te = eval_metrics["train"][metric], eval_metrics["test"][metric]
    gap_rows.append({"métrique": metric, "train": tr, "test": te, "écart": tr - te})

gap_df = pd.DataFrame(gap_rows).round(4)
gap_df


In [ ]:
# Cellule 4.13 — Conclusion sur la généralisation
acc_gap = eval_metrics["train"]["accuracy"] - eval_metrics["test"]["accuracy"]
verdict = (
    "✅ Généralisation correcte (écart < 3 points)"
    if acc_gap < 0.03
    else "⚠️ Surapprentissage modéré"
    if acc_gap < 0.08
    else "❌ Surapprentissage important : renforcer l'augmentation / le dropout"
)
print(f"Accuracy train : {eval_metrics['train']['accuracy']:.4f}")
print(f"Accuracy test  : {eval_metrics['test']['accuracy']:.4f}")
print(f"Écart           : {acc_gap:+.4f} ({acc_gap * 100:+.2f} points)")
print(f"\n{verdict}")

fig, ax = plt.subplots(figsize=(7, 4))
width = 0.35
x = np.arange(len(["accuracy", "precision", "recall", "f1"]))
ax.bar(x - width / 2, [eval_metrics["train"][m] for m in ["accuracy", "precision", "recall", "f1"]],
       width, label="train", color="tab:blue")
ax.bar(x + width / 2, [eval_metrics["test"][m] for m in ["accuracy", "precision", "recall", "f1"]],
       width, label="test", color="tab:orange")
ax.set_xticks(x, ["accuracy", "precision", "recall", "F1"])
ax.set_ylim(0, 1.05)
ax.set_title("Performances train vs test", weight="bold")
ax.legend()
plt.tight_layout()
plt.show()


### 1.33 — Sauvegarde du modèle entraîné et des artefacts

Sont sauvegardés :

* `models/classification/brain_tumor_googlenet.keras` — **modèle Keras complet**
  (prétraitement et augmentation inclus : directement exploitable par l'API) ;
* `models/classification/metadata.json` — classes, hyperparamètres, métriques ;
* `artifacts/reports/…` — matrice de confusion, courbes, rapport de classification.

In [ ]:
# Cellule 4.14 — Sauvegarde du modèle et des métadonnées
MODEL_DIR = MODELS_DIR / "classification"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
MODEL_PATH = MODEL_DIR / "brain_tumor_googlenet.keras"

model.save(MODEL_PATH)

metadata = {
    "task": "brain_tumor_classification",
    "architecture": "GoogLeNet (Inception v1) - transfer learning torchvision IMAGENET1K_V1",
    "framework": f"tensorflow {tf.__version__} / keras {keras.__version__}",
    "input_shape": [IMAGE_SIZE, IMAGE_SIZE, 3],
    "classes": class_names,
    "class_display": {c: CLASS_DISPLAY.get(c, c) for c in class_names},
    "hyperparameters": {
        "optimizer": "Adam",
        "loss": "categorical_crossentropy",
        "learning_rate_phase_a": LR_PHASE_A,
        "learning_rate_phase_b": LR_PHASE_B,
        "batch_size": BATCH_SIZE,
        "epochs_phase_a": EPOCHS_A,
        "epochs_phase_b": EPOCHS_B,
        "augmentation": ["RandomFlip(horizontal)", "RandomGaussianBlur", "GaussianNoise"],
        "dropout": 0.5,
        "dense_units": 128,
        "seed": SEED,
    },
    "metrics": {
        "train": eval_metrics["train"],
        "validation": eval_metrics["valid"],
        "test": eval_metrics["test"],
    },
    "training_seconds": {"phase_a": phase_a_seconds, "phase_b": phase_b_seconds},
    "saved_at": time.strftime("%Y-%m-%dT%H:%M:%S"),
}
(MODEL_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

print(f"Modèle    : {MODEL_PATH}  ({MODEL_PATH.stat().st_size / 1e6:.1f} Mo)")
print(f"Métadonnées: {MODEL_DIR / 'metadata.json'}")


In [ ]:
# Cellule 4.15 — Export des figures et rapports
report_dir = REPORTS_DIR / "classification"
report_dir.mkdir(parents=True, exist_ok=True)

# Matrice de confusion
fig, ax = plt.subplots(figsize=(6.5, 5.5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_names, yticklabels=class_names,
            ax=ax, cbar=False)
ax.set_title("Matrice de confusion — TEST", weight="bold")
ax.set_xlabel("Classe prédite")
ax.set_ylabel("Classe réelle")
fig.tight_layout()
fig.savefig(report_dir / "confusion_matrix_test.png", dpi=150)
plt.close(fig)

# Courbes
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(epochs, history["loss"], label="train")
axes[0].plot(epochs, history["val_loss"], label="validation")
axes[0].set_title("Perte", weight="bold")
axes[0].set_xlabel("Époque")
axes[0].legend()
axes[1].plot(epochs, history["accuracy"], label="train")
axes[1].plot(epochs, history["val_accuracy"], label="validation")
axes[1].set_title("Accuracy", weight="bold")
axes[1].set_xlabel("Époque")
axes[1].legend()
fig.tight_layout()
fig.savefig(report_dir / "training_curves.png", dpi=150)
plt.close(fig)

# Rapport texte
report_txt = classification_report(y_true, y_pred, target_names=class_names, digits=4, zero_division=0)
(report_dir / "classification_report_test.txt").write_text(report_txt, encoding="utf-8")

np.save(report_dir / "confusion_matrix_test.npy", cm)
(report_dir / "evaluation_metrics.json").write_text(
    json.dumps({"train": eval_metrics["train"], "validation": eval_metrics["valid"], "test": eval_metrics["test"]},
               indent=2, ensure_ascii=False),
    encoding="utf-8",
)

print("Artefacts :")
for p in sorted(report_dir.iterdir()):
    print(f"  {p.name:<34s} {p.stat().st_size / 1024:>8.1f} Ko")


### 1.34 — Suivi de l'expérience avec MLflow

L'expérience enregistre :

* les **hyperparamètres** (`mlflow.log_params`) ;
* les **métriques** train / validation / test (`mlflow.log_metrics`) ;
* les **artefacts** : matrices de confusion, courbes, rapport, métadonnées, modèle ;
* le **modèle Keras** (`mlflow.keras.log_model`) pour réutilisation ultérieure.

Si le serveur MLflow est injoignable, le suivi bascule automatiquement sur un backend
local (`./mlruns`) afin que le notebook reste exécutable.

In [ ]:
# Cellule 4.16 — Connexion à MLflow
mlflow_available = True
try:
    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
    print(f"MLflow connecté : {MLFLOW_TRACKING_URI}")
    print(f"Experiment     : {MLFLOW_EXPERIMENT_NAME}")
    client = mlflow.tracking.MlflowClient()
    exp = client.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)
    print(f"Experiment ID  : {exp.experiment_id if exp else 'créé à la volée'}")
except Exception as exc:  # noqa: BLE001
    mlflow_available = False
    mlflow.set_tracking_uri(str(PROJECT_ROOT / "mlruns"))
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)
    print(f"⚠️  MLflow distant indisponible ({exc}) — suivi local activé : {mlflow.get_tracking_uri()}")


In [ ]:
# Cellule 4.17 — Enregistrement du run MLflow
RUN_NAME = "classification-googlenet-phaseAB"

with mlflow.start_run(run_name=RUN_NAME) as run:
    mlflow.log_params({
        "architecture": "googlenet",
        "weights_init": "imagenet (torchvision IMAGENET1K_V1)",
        "optimizer": "adam",
        "loss": "categorical_crossentropy",
        "lr_phase_a": LR_PHASE_A,
        "lr_phase_b": LR_PHASE_B,
        "batch_size": BATCH_SIZE,
        "epochs_phase_a": EPOCHS_A,
        "epochs_phase_b": EPOCHS_B,
        "image_size": IMAGE_SIZE,
        "dense_units": 128,
        "dropout": 0.5,
        "augmentation": "RandomFlip+RandomGaussianBlur+GaussianNoise",
        "seed": SEED,
        "n_classes": len(class_names),
    })

    for split, metrics in eval_metrics.items():
        for metric_name, value in metrics.items():
            mlflow.log_metric(f"{split}_{metric_name}", value)

    mlflow.log_metric("train_test_accuracy_gap", acc_gap)

    mlflow.log_artifact(str(report_dir / "confusion_matrix_test.png"), artifact_path="figures")
    mlflow.log_artifact(str(report_dir / "training_curves.png"), artifact_path="figures")
    mlflow.log_artifact(str(report_dir / "classification_report_test.txt"), artifact_path="reports")
    mlflow.log_artifact(str(report_dir / "evaluation_metrics.json"), artifact_path="reports")
    mlflow.log_artifact(str(MODEL_DIR / "metadata.json"), artifact_path="reports")
    mlflow.log_artifact(str(CHECKPOINT_DIR / "googlenet_phaseA.keras"), artifact_path="checkpoints")

    mlflow.keras.log_model(
        keras_model=model,
        artifact_path="model",
        registered_model_name=None,
        input_example=np.zeros((1, IMAGE_SIZE, IMAGE_SIZE, 3), dtype="float32"),
    )

    run_id = run.info.run_id
    print(f"Run MLflow enregistré : {run_id}")

mlflow_uri = mlflow.get_tracking_uri()
print(f"Suivi disponible sur {mlflow_uri} -> run {run_id}")


In [ ]:
# Cellule 4.18 — Récapitulatif des runs MLflow (comparaison des expériences)
client = mlflow.tracking.MlflowClient()
experiment = client.get_experiment_by_name(MLFLOW_EXPERIMENT_NAME)

if experiment is None:
    print("Aucune expérience enregistrée.")
else:
    runs = client.search_runs([experiment.experiment_id], order_by=["attributes.start_time DESC"], max_results=20)
    runs_df = pd.DataFrame(
        [
            {
                "run_id": r.info.run_id[:8],
                "run_name": r.data.tags.get("mlflow.runName", "?"),
                "test_accuracy": r.data.metrics.get("test_accuracy"),
                "test_f1": r.data.metrics.get("test_f1"),
                "test_recall": r.data.metrics.get("test_recall"),
                "test_precision": r.data.metrics.get("test_precision"),
                "status": r.info.status,
            }
            for r in runs
        ]
    )
    print(f"{len(runs_df)} run(s) dans l'expérience « {MLFLOW_EXPERIMENT_NAME} »")
    if len(runs_df):
        runs_df.sort_values("test_accuracy", ascending=False)


### 1.35 — Récapitulatif final

L'étape 1 (classification) est terminée :

* **données** : 7 023 images contrôlées, split stratifié 70/15/15, dataset nettoyé RGB 224×224 ;
* **modèle** : GoogLeNet pré-entraîné (transfert des poids ImageNet vérifié numériquement),
  classifieur fully connected adapté à 4 classes, augmentation flip/flou/bruit ;
* **entraînement** : 2 phases (extraction de caractéristiques puis fine-tuning) ;
* **évaluation** : accuracy / precision / recall / F1, matrices de confusion, analyse
  de l'écart train/test ;
* **livrables** : modèle `.keras`, métadonnées, rapports, run MLflow.

**Suite du projet** : étape 2 (détection YOLOv8), étape 3 (MLflow), étape 4 (Airflow),
étape 5 (API FastAPI + Streamlit).